# 08 · Analysis and figures

**Purpose.** Correlate leakage with the generalisation gap and the loss ratio across all target points, draw figures F1–F7, export the results table and fill the README results summary.

**Inputs.** The complete database (notebooks 00–07) and the CIFAR pool (for F6).

**Outputs.** `results/figures/F1…F7.{pdf,png}` (300 dpi), `results/reports/results_tables.csv`, README results block.

**Syllabus topics.** Statistical analysis of classifiers; energy-efficient edge models (F7).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import matplotlib.pyplot as plt

from mia import db, plots
from mia.config import DB_PATH, RAW_DIR, REPORTS_DIR, ROOT, load_experiment
from mia.data import load_pool
from mia.metrics import spearman_with_permutation
from mia.training import headline_model_id

exp = load_experiment()
conn = db.connect(DB_PATH)
pts = db.leakage_vs_gap_df(conn).drop_duplicates("model_id")
rho_gap, p_gap = spearman_with_permutation(pts["gap_acc"], pts["auc"], exp.permutations, exp.bootstrap_seed)
lr = pts[pts["loss_ratio"].notna()]
rho_lr, p_lr = spearman_with_permutation(lr["loss_ratio"], lr["auc"], exp.permutations, exp.bootstrap_seed)
print(f"{len(pts)} target points")
print(f"AUC vs accuracy gap: Spearman rho = {rho_gap:.3f}, permutation p = {p_gap:.5f}")
print(f"AUC vs loss ratio:   Spearman rho = {rho_lr:.3f}, permutation p = {p_lr:.5f}")

In [ ]:
plots.apply_style()
for fig, stem in [(plots.f1_auc_vs_gap(pts, rho_gap, p_gap), "F1_auc_vs_gap"),
                  (plots.f2_auc_vs_loss_ratio(lr, rho_lr, p_lr), "F2_auc_vs_loss_ratio"),
                  (plots.f3_early_stopping(db.sweep_df(conn, "early_stopping")), "F3_early_stopping"),
                  (plots.f4_label_smoothing(db.sweep_df(conn, "label_smoothing")), "F4_label_smoothing")]:
    plots.save_figure(fig, stem)
    plt.show()

In [ ]:
runs = db.attack_runs_df(conn)
heads = {cid: headline_model_id(exp, cid) for cid in ("C1", "C2", "C3", "C4", "C5", "C6", "C7")}
best = pts.set_index("model_id")


def run_id(mid, attack):
    r = runs[(runs["target_model_id"] == mid) & (runs["attack"] == attack)].sort_values("auc", ascending=False)
    return int(r["attack_run_id"].iloc[0])


curves = {}
for cid, mid in heads.items():
    attack = best.loc[mid, "attack"]
    s = db.sample_scores_df(conn, run_id(mid, attack))
    curves[f"{cid} ({attack}, AUC {best.loc[mid, 'auc']:.3f})"] = (s["score"].to_numpy(), s["is_member"].to_numpy(), cid)
fig = plots.f5_roc_curves(curves)
plots.save_figure(fig, "F5_roc_loglog")
plt.show()

images, labels, _ = load_pool(RAW_DIR)
panels = {heads[c]: db.sample_scores_df(conn, run_id(heads[c], "shokri_lda")) for c in ("C2", "C7")}
fig = plots.f6_exposed_members(images, labels, panels)
plots.save_figure(fig, "F6_exposed_members")
plt.show()

f7 = best.loc[list(heads.values())].reset_index()
fig = plots.f7_cost_vs_leakage(f7.merge(runs[["target_model_id", "model_bytes"]].drop_duplicates(),
                                        left_on="model_id", right_on="target_model_id"))
plots.save_figure(fig, "F7_cost_vs_leakage")
plt.show()

## Results table and README summary

In [ ]:
import re

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
runs.to_csv(REPORTS_DIR / "results_tables.csv", index=False)
print("results table:", REPORTS_DIR / "results_tables.csv", runs.shape)

hi, lo = pts.loc[pts["auc"].idxmax()], pts.loc[pts["auc"].idxmin()]
rows = "\n".join(f"| {cid} | `{mid}` | {best.loc[mid, 'test_acc'] * 100:.1f}% | {best.loc[mid, 'gap_acc'] * 100:.1f} pp | "
                  f"{best.loc[mid, 'attack']} | {best.loc[mid, 'auc']:.3f} | {best.loc[mid, 'tpr_at_01pct_fpr'] * 100:.2f}% | "
                  f"{best.loc[mid, 'severity']} |" for cid, mid in heads.items())
summary = (f"<!-- RESULTS:START -->\n"
           f"- Spearman ρ (AUC vs train–test gap) = **{rho_gap:.3f}** (permutation p = {p_gap:.4f}, {len(pts)} target points)\n"
           f"- Spearman ρ (AUC vs loss ratio) = **{rho_lr:.3f}** (p = {p_lr:.4f})\n"
           f"- Highest AUC: **{hi['auc']:.3f}** (`{hi['model_id']}`, {hi['attack']}); lowest AUC: **{lo['auc']:.3f}** "
           f"(`{lo['model_id']}`, {lo['attack']})\n\n"
           f"| Config | Model | Test acc | Gap | Best attack | AUC | TPR @ 0.1% FPR | Severity |\n"
           f"|---|---|---:|---:|---|---:|---:|---|\n{rows}\n<!-- RESULTS:END -->")
readme = ROOT / "README.md"
text = readme.read_text(encoding="utf-8")
readme.write_text(re.sub(r"<!-- RESULTS:START -->.*?<!-- RESULTS:END -->", lambda _: summary, text, flags=re.S), encoding="utf-8")
print(summary)

## Results
The printed block (also written into README.md) is the headline result: the correlation between leakage and the generalisation gap, and the best attack per configuration. Figures F1–F7 are in `results/figures/`.